Imports

In [72]:
import pandas as pd
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

from scipy.sparse import hstack

Load Data

In [73]:
train_df = pd.read_parquet("../data/processed/priority_train.parquet")
test_df  = pd.read_parquet("../data/processed/priority_test.parquet")

Split Features and Target

In [74]:
X_train = train_df.drop(columns=["priority"])
y_train = train_df["priority"]

X_test = test_df.drop(columns=["priority"])
y_test = test_df["priority"]


Text Cleaning Function(
Remove masked tokens like XXXX)

In [75]:
def remove_masked_tokens(text):
    return re.sub(
        r"\s+",
        " ",
        re.sub(r"\b\w*x{2,}\w*\b", " ", text, flags=re.IGNORECASE)
    ).strip()


X_train["Consumer complaint narrative"] = \
    X_train["Consumer complaint narrative"].apply(remove_masked_tokens)

X_test["Consumer complaint narrative"] = \
    X_test["Consumer complaint narrative"].apply(remove_masked_tokens)

Style Feature Extraction

In [76]:
def extract_style_features(text):

    if not isinstance(text, str):
        return 0, 0

    repeated_exclam = len(re.findall(r"!{2,}", text))

    words = text.split()
    caps_words = sum(1 for w in words if w.isupper())
    caps_ratio = caps_words / len(words) if words else 0

    return repeated_exclam, caps_ratio

Train features

In [77]:
features = X_train["Consumer complaint narrative"].apply(extract_style_features)

features_df = features.apply(pd.Series)
features_df.columns = ["exclam_flag", "caps_ratio"]

X_train = X_train.join(features_df)

Test features

In [78]:
features = X_test["Consumer complaint narrative"].apply(extract_style_features)

features_df = features.apply(pd.Series)
features_df.columns = ["exclam_flag", "caps_ratio"]

X_test = X_test.join(features_df)

Text Vectorization

In [79]:
vectorizer = CountVectorizer(stop_words="english")

X_text_train = vectorizer.fit_transform(
    X_train["Consumer complaint narrative"]
)

X_text_test = vectorizer.transform(
    X_test["Consumer complaint narrative"]
)

Combine Text + Extra Features

In [81]:
X_extra_train = X_train[["exclam_flag", "caps_ratio"]].values * 5
X_extra_test  = X_test[["exclam_flag", "caps_ratio"]].values * 5

X_train_vec = hstack([X_text_train, X_extra_train])
X_test_vec  = hstack([X_text_test, X_extra_test])

Train Model

In [82]:
nb = MultinomialNB()

nb.fit(X_train_vec, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Predictions

In [83]:
y_pred = nb.predict(X_test_vec)

 Evaluation

In [84]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

   immediate       0.64      0.70      0.67     65581
     regular       0.80      0.81      0.80    168223
    same_day       0.66      0.54      0.59     46809

    accuracy                           0.74    280613
   macro avg       0.70      0.68      0.69    280613
weighted avg       0.74      0.74      0.74    280613

